# TTS Playground — Try Out the TTS Models

In [ ]:
!pip install -U -q kittenml soundfile
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision torchcodec

## TTS Model 1 — KittenTTS

In [ ]:
from kittenml import KittenTTS

kitten = KittenTTS("KittenML/kitten-tts-2-mini")
print(kitten.available_voices)

for v in kitten.voices("en"):
    print(v["name"], "-", v["description"])

for v in kitten.voices("de"):
    print(v["name"], "-", v["description"])

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

text = "Hello there, this is Kitten TTS 2 Mini."   # <-- change the text
voice = "Eleanor"   # <-- the voice sets the language; e.g. "Lena" or "Felix" for German
speed = 1.0   # <-- e.g. 1.1 for slightly faster speech

audio = kitten.generate(text, voice=voice, speed=speed)
sf.write("tts_exercise/kitten_out.wav", audio, 24000)

display(Audio("tts_exercise/kitten_out.wav"))

In [ ]:
# generate by language (uses the language's first voice)
text = "Guten Morgen! Wie geht es dir?"   # <-- change the text
language = "de"

audio = kitten.generate(text, language=language)
sf.write("tts_exercise/kitten_out_de.wav", audio, 24000)

display(Audio("tts_exercise/kitten_out_de.wav"))

## TTS Model 2 — Audio8 (Voice Cloning)

In [ ]:
import torch
import soundfile as sf
from transformers import AutoModel, AutoProcessor

model_id = "Edge0/Audio8-TTS-Realistic-Air"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=dtype).eval().to(device)
print(f"Model loaded on {model.device} with dtype {model.dtype}")

In [ ]:
REFERENCE_WAV = "tts_exercise/reference.wav"   # <-- adapt: a short reference recording
REFERENCE_TEXT = "The exact transcript of the reference recording."   # <-- its transcript

from IPython.display import Audio, display

# listen to the reference first: the clone should sound like this speaker
ref_waveform, ref_sr = sf.read(REFERENCE_WAV)
display(Audio(ref_waveform, rate=ref_sr))

In [ ]:
text = "Welcome to Audio8 TTS. It is wonderful to hear from you."   # <-- change the text

inputs = processor(
    text=[text],
    reference_audio=[REFERENCE_WAV],
    reference_text=[REFERENCE_TEXT],
    return_tensors="pt",
)
inputs = {name: value.to(device) for name, value in inputs.items()}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.8,
        top_p=0.95,
        top_k=50,
        do_sample=True,
        return_dict_in_generate=True,
    )
    waveforms, waveform_lengths = model.decode_audio(output.codes)

audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
sf.write("tts_exercise/audio8_out.wav", audio, model.config.codec_sample_rate)

display(Audio("tts_exercise/audio8_out.wav"))

## Experiment Ideas

In [ ]:
# Compare the two models on the same sentence:
#   - naturalness, speaker similarity (Audio8), intelligibility
#   - how do both handle numbers, abbreviations, foreign words?
#   - KittenTTS: how do the voices differ across languages?
#   - Audio8: what changes with a different reference speaker?